<a href="https://colab.research.google.com/github/Zyu-Peng/learning_code/blob/main/ESMFold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**ESMFold**
for more details see: [Github](https://github.com/facebookresearch/esm/tree/main/esm), [Preprint](https://www.biorxiv.org/content/10.1101/2022.07.20.500902v1)

#### **Tips and Instructions**
- click the little ▶ play icon to the left of each cell below.
- use "/" to specify chainbreaks, (eg. sequence="AAA/AAA")
- for homo-oligomeric predictions, set copies > 1
- See [experimental notebook](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/beta/ESMFold_advanced.ipynb) for more advanced options (like sampling).

#### **Colab Limitations**
- For short monomeric proteins under the length 400, consider using [ESMFold API](https://esmatlas.com/resources?action=fold) (no need for GPU, super fast!)
- On Tesla T4 (typical free colab GPU), max total length ~ 900

In [1]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/drive')
from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

Mounted at /content/drive


In [3]:
%%time
#@title install
#@markdown install ESMFold, OpenFold and download Params (~2min 30s)
version = "1" # @param ["0", "1"]
model_name = "esmfold_v0.model" if version == "0" else "esmfold.model"
import os, time
if not os.path.isfile(model_name):
  # download esmfold params
  os.system("apt-get install aria2 -qq")
  os.system(f"aria2c -q -x 16 https://colabfold.steineggerlab.workers.dev/esm/{model_name} &")

  if not os.path.isfile("finished_install"):
    # install libs
    print("installing libs...")
    os.system("pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol modelcif")
    os.system("pip install -q git+https://github.com/NVIDIA/dllogger.git")

    print("installing openfold...")
    # install openfold
    os.system(f"pip install -q git+https://github.com/sokrypton/openfold.git")

    print("installing esmfold...")
    # install esmfold
    os.system(f"pip install -q git+https://github.com/sokrypton/esm.git")
    os.system("touch finished_install")

  # wait for Params to finish downloading...
  while not os.path.isfile(model_name):
    time.sleep(5)
  if os.path.isfile(f"{model_name}.aria2"):
    print("downloading params...")
  while os.path.isfile(f"{model_name}.aria2"):
    time.sleep(5)

installing libs...
installing openfold...
installing esmfold...
CPU times: user 4.4 ms, sys: 8.6 ms, total: 13 ms
Wall time: 3min 13s


In [2]:
#@title Input protein sequence, then hit `Runtime` -> `Run all`

input_dir = '/content/drive/MyDrive/data' #@param {type:"string"}
result_dir = '/content/drive/MyDrive/train_pos' #@param {type:"string"}

# number of models to use
#@markdown ---
#@markdown ### Advanced settings
msa_mode = "MMseqs2 (UniRef+Environmental)" #@param ["MMseqs2 (UniRef+Environmental)", "MMseqs2 (UniRef only)","single_sequence","custom"]
num_models = 5 #@param [1,2,3,4,5] {type:"raw"}
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
stop_at_score = 100 #@param {type:"string"}
#@markdown - early stop computing models once score > threshold (avg. plddt for "structures" and ptmscore for "complexes")
use_custom_msa = False
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
use_amber = num_relax > 0
relax_max_iterations = 200 #@param [0,200,2000] {type:"raw"}
use_templates = False #@param {type:"boolean"}
do_not_overwrite_results = True #@param {type:"boolean"}
zip_results = False #@param {type:"boolean"}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 1. 基础参数配置
import os
import re
import numpy as np
import torch
import gc
import esm

# 基础路径配置（修改为你的Drive路径）
input_dir = '/content/drive/MyDrive/data' #@param {type:"string"}
result_dir = '/content/drive/MyDrive/train_pos' #@param {type:"string"}

#@markdown ---
#@markdown ### Advanced settings
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
do_not_overwrite_results = True #@param {type:"boolean"}

# 创建结果目录（确保存在）
os.makedirs(result_dir, exist_ok=True)

#@title 2. 运行ESMFold预测蛋白结构（仅输出PDB）
%%time
# ===================== 核心函数定义 =====================
def get_simple_seq_name(seq, max_len=30):
    """生成简洁的序列命名：保留前max_len个字符+长度，避免非法字符"""
    # 清理序列（仅保留A-Z和:，用于多聚体）
    clean_seq = re.sub("[^A-Z:]", "", seq.upper())
    # 长序列截断，避免文件名过长
    short_seq = clean_seq[:max_len] if len(clean_seq) > max_len else clean_seq
    # 替换冒号（多聚体分隔符）为下划线，避免文件名非法
    short_seq = short_seq.replace(":", "_")
    # 最终命名：短序列_长度.pdb
    seq_len = len(clean_seq.replace(":", ""))  # 总氨基酸数（不含分隔符）
    return f"{short_seq}_len{seq_len}"

# ===================== 任务配置 =====================
# 输入蛋白序列（替换为你的目标序列）
sequence = "GWSTELEKHREELKEFLKKEGITNVEIRIDNGRLEVRVEGGTERLKRFLEELRQKLEKKGYTVDIKIE" #@param {type:"string"}
# 清理序列：仅保留大写字母和多聚体分隔符:
sequence = re.sub("[^A-Z:]", "", sequence.replace("/",":").upper())
sequence = re.sub(":+",":",sequence).strip(":")

copies = 1 #@param {type:"integer"}
copies = 1 if copies <= 0 else copies
sequence = ":".join([sequence] * copies)  # 复制序列（用于多聚体）

chain_linker = 25  # 链间连接序列长度
# 生成简洁的序列文件名
seq_filename = get_simple_seq_name(sequence)
print(f"序列命名: {seq_filename}")

# 解析序列信息
seqs = sequence.split(":")
total_length = sum([len(s) for s in seqs])
print(f"蛋白总长度: {total_length}")

# ===================== 加载ESMFold模型 =====================
model = None
model_name_ = ""

# 加载模型（重复运行时释放显存）
if "model" not in locals() or model_name_ != "esmfold_v1":
    if "model" in locals():
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 加载ESMFold v1模型
    print("加载ESMFold模型...")
    model, alphabet = esm.pretrained.esmfold_v1()
    model = model.eval().cuda() if torch.cuda.is_available() else model.eval()
    model.requires_grad_(False)
    model_name_ = "esmfold_v1"

# 根据序列长度调整chunk size（优化显存）
chunk_size = 64 if total_length > 700 else 128
model.set_chunk_size(chunk_size)

# ===================== 结构预测 & 保存PDB =====================
torch.cuda.empty_cache()
print("开始结构预测...")
try:
    # 执行预测
    output = model.infer(
        sequence,
        num_recycles=num_recycles,
        chain_linker="X"*chain_linker,
        residue_index_offset=512
    )

    # 生成PDB内容
    pdb_str = model.output_to_pdb(output)[0]

    # 计算评估指标（仅打印，可选）
    output_np = {k: v.cpu().numpy() for k, v in output.items()}
    ptm_score = output_np["ptm"][0]
    avg_plddt = output_np["plddt"][0,...,1].mean()
    print(f"PTM分数: {ptm_score:.3f} | 平均pLDDT分数: {avg_plddt:.3f}")

    # 保存PDB文件（按序列命名）
    pdb_path = os.path.join(result_dir, f"{seq_filename}.pdb")

    # 处理覆盖逻辑
    if do_not_overwrite_results and os.path.exists(pdb_path):
        idx = 1
        while os.path.exists(os.path.join(result_dir, f"{seq_filename}_{idx}.pdb")):
            idx += 1
        pdb_path = os.path.join(result_dir, f"{seq_filename}_{idx}.pdb")

    # 写入PDB文件
    with open(pdb_path, "w") as f:
        f.write(pdb_str)

    print(f"\n✅ PDB文件已保存至：")
    print(f"   {pdb_path}")

except Exception as e:
    print(f"\n❌ 预测出错: {str(e)}")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
#@title display (optional) {run: "auto"}
import py3Dmol
pymol_color_list = ["#33ff33","#00ffff","#ff33cc","#ffff00","#ff9999","#e5e5e5","#7f7fff","#ff7f00",
                    "#7fff7f","#199999","#ff007f","#ffdd5e","#8c3f99","#b2b2b2","#007fff","#c4b200",
                    "#8cb266","#00bfbf","#b27f7f","#fcd1a5","#ff7f7f","#ffbfdd","#7fffff","#ffff7f",
                    "#00ff7f","#337fcc","#d8337f","#bfff3f","#ff7fff","#d8d8ff","#3fffbf","#b78c4c",
                    "#339933","#66b2b2","#ba8c84","#84bf00","#b24c66","#7f7f7f","#3f3fa5","#a5512b"]

def show_pdb(pdb_str, show_sidechains=False, show_mainchains=False,
             color="pLDDT", chains=None, vmin=50, vmax=90,
             size=(800,480), hbondCutoff=4.0,
             Ls=None,
             animate=False):

  if chains is None:
    chains = 1 if Ls is None else len(Ls)
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width=size[0], height=size[1])
  if animate:
    view.addModelsAsFrames(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  else:
    view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  if color == "pLDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':vmin,'max':vmax}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})
  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                  {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  view.zoomTo()
  if animate: view.animate()
  return view

color = "confidence" #@param ["confidence", "rainbow", "chain"]
if color == "confidence": color = "pLDDT"
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}
show_pdb(pdb_str, color=color,
         show_sidechains=show_sidechains,
         show_mainchains=show_mainchains,
         Ls=lengths).show()

In [ ]:
#@title plot confidence (optional)

dpi = 100 #@param {type:"integer"}

def plot_ticks(Ls):
  Ln = sum(Ls)
  L_prev = 0
  for L_i in Ls[:-1]:
    L = L_prev + L_i
    L_prev += L_i
    plt.plot([0,Ln],[L,L],color="black")
    plt.plot([L,L],[0,Ln],color="black")
  ticks = np.cumsum([0]+Ls)
  ticks = (ticks[1:] + ticks[:-1])/2
  plt.yticks(ticks,alphabet_list[:len(ticks)])

def plot_confidence(O, Ls=None, dpi=100):
  if "lm_contacts" in O:
    plt.figure(figsize=(20,4), dpi=dpi)
    plt.subplot(1,4,1)
  else:
    plt.figure(figsize=(15,4), dpi=dpi)
    plt.subplot(1,3,1)

  plt.title('Predicted lDDT')
  plt.plot(O["plddt"])
  if Ls is not None:
    L_prev = 0
    for L_i in Ls[:-1]:
      L = L_prev + L_i
      L_prev += L_i
      plt.plot([L,L],[0,100],color="black")
  plt.xlim(0,O["plddt"].shape[0])
  plt.ylim(0,100)
  plt.ylabel('plDDT')
  plt.xlabel('position')
  plt.subplot(1,4 if "lm_contacts" in O else 3,2)

  plt.title('Predicted Aligned Error')
  Ln = O["pae"].shape[0]
  plt.imshow(O["pae"],cmap="bwr",vmin=0,vmax=30,extent=(0, Ln, Ln, 0))
  if Ls is not None and len(Ls) > 1: plot_ticks(Ls)
  plt.colorbar()
  plt.xlabel('Scored residue')
  plt.ylabel('Aligned residue')

  if "lm_contacts" in O:
    plt.subplot(1,4,3)
    plt.title("contacts from LM")
    plt.imshow(O["lm_contacts"],cmap="Greys",vmin=0,vmax=1,extent=(0, Ln, Ln, 0))
    if Ls is not None and len(Ls) > 1: plot_ticks(Ls)
    plt.subplot(1,4,4)
  else:
    plt.subplot(1,3,3)
  plt.title("contacts from Structure Module")
  plt.imshow(O["sm_contacts"],cmap="Greys",vmin=0,vmax=1,extent=(0, Ln, Ln, 0))
  if Ls is not None and len(Ls) > 1: plot_ticks(Ls)
  return plt

plot_confidence(O, Ls=lengths, dpi=dpi)
plt.savefig(f'{prefix}.png',bbox_inches='tight')
plt.show()

In [ ]:
#@title download predictions
from google.colab import files
os.system(f"zip {ID}.zip {ID}/*")
files.download(f'{ID}.zip')